### DenseNet model 
@GalDude33 GalDude33 Adapted DenseNet and DenseNetFCN to 3D inputs (3D convolutions and et… 2dddf18 on Feb 26
946 lines (809 sloc) 43.1 KB

In [0]:
!pip install git+https://www.github.com/keras-team/keras-contrib.git

  Cloning https://www.github.com/keras-team/keras-contrib.git to /tmp/pip-req-build-b6j7z88e
  Running command git clone -q https://www.github.com/keras-team/keras-contrib.git /tmp/pip-req-build-b6j7z88e
  Created wheel for keras-contrib: filename=keras_contrib-2.0.8-cp36-none-any.whl size=101065 sha256=731c7873a3e15daead1e9b2cf75a815e093468b7d84632e6d06dfceed18deb72
  Stored in directory: /tmp/pip-ephem-wheel-cache-o8718px8/wheels/11/27/c8/4ed56de7b55f4f61244e2dc6ef3cdbaff2692527a2ce6502ba
Successfully built keras-contrib


In [0]:
'''DenseNet and DenseNet-FCN models for Keras.
DenseNet is a network architecture where each layer is directly connected
to every other layer in a feed-forward fashion (within each dense block).
For each layer, the feature maps of all preceding layers are treated as
separate inputs whereas its own feature maps are passed on as inputs to
all subsequent layers. This connectivity pattern yields state-of-the-art
accuracies on CIFAR10/100 (with or without data augmentation) and SVHN.
On the large scale ILSVRC 2012 (ImageNet) dataset, DenseNet achieves a
similar accuracy as ResNet, but using less than half the amount of
parameters and roughly half the number of FLOPs.
DenseNets can be extended to image segmentation tasks as described in the
paper "The One Hundred Layers Tiramisu: Fully Convolutional DenseNets for
Semantic Segmentation". Here, the dense blocks are arranged and concatenated
with long skip connections for state of the art performance on the CamVid dataset.
# Reference
- [Densely Connected Convolutional Networks](https://arxiv.org/pdf/1608.06993.pdf)
- [The One Hundred Layers Tiramisu: Fully Convolutional DenseNets for Semantic
   Segmentation](https://arxiv.org/pdf/1611.09326.pdf)
This implementation is based on the following reference code:
 - https://github.com/gpleiss/efficient_densenet_pytorch
 - https://github.com/liuzhuang13/DenseNet
'''
from __future__ import absolute_import
from __future__ import division
from __future__ import print_function

import keras
import keras.backend as K
from keras.engine.topology import get_source_inputs
from keras.layers import Activation
from keras.layers import AveragePooling3D
from keras.layers import BatchNormalization
from keras.layers import Conv3D
from keras.layers import Conv3DTranspose
from keras.layers import Dense
from keras.layers import Dropout
from keras.layers import GlobalAveragePooling3D
from keras.layers import GlobalMaxPooling3D
from keras.layers import Input
from keras.layers import MaxPooling3D
from keras.layers import Reshape
from keras.layers import UpSampling3D
from keras.layers import concatenate
from keras.models import Model
from keras.regularizers import l2
from keras_contrib.layers import SubPixelUpscaling
import keras.optimizers as OPTS

Using TensorFlow backend.


In [0]:
def name_or_none(prefix, name):
    return prefix + name if (prefix is not None and name is not None) else None


def __conv_block(ip, nb_filter, bottleneck=False, dropout_rate=None,
                 weight_decay=1e-4, block_prefix=None):
    '''
    Adds a convolution layer (with batch normalization and relu),
    and optionally a bottleneck layer.
    # Arguments
        ip: Input tensor
        nb_filter: integer, the dimensionality of the output space
            (i.e. the number output of filters in the convolution)
        bottleneck: if True, adds a bottleneck convolution block
        dropout_rate: dropout rate
        weight_decay: weight decay factor
        block_prefix: str, for unique layer naming
     # Input shape
        4D tensor with shape:
        `(samples, channels, rows, cols)` if data_format='channels_first'
        or 4D tensor with shape:
        `(samples, rows, cols, channels)` if data_format='channels_last'.
    # Output shape
        4D tensor with shape:
        `(samples, filters, new_rows, new_cols)` if data_format='channels_first'
        or 4D tensor with shape:
        `(samples, new_rows, new_cols, filters)` if data_format='channels_last'.
        `rows` and `cols` values might have changed due to stride.
    # Returns
        output tensor of block
    '''
    with K.name_scope('ConvBlock'):
        concat_axis = 1 if K.image_data_format() == 'channels_first' else -1

        x = BatchNormalization(axis=concat_axis, epsilon=1.1e-5,
                               name=name_or_none(block_prefix, '_bn'))(ip)
        x = Activation('relu')(x)

        if bottleneck:
            inter_channel = nb_filter * 4

            x = Conv3D(inter_channel, (1, 1, 1), kernel_initializer='he_normal',
                       padding='same', use_bias=False,
                       kernel_regularizer=l2(weight_decay),
                       name=name_or_none(block_prefix, '_bottleneck_Conv3D'))(x)
            x = BatchNormalization(axis=concat_axis, epsilon=1.1e-5,
                                   name=name_or_none(block_prefix, '_bottleneck_bn'))(x)
            x = Activation('relu')(x)

        x = Conv3D(nb_filter, (3, 3, 3), kernel_initializer='he_normal', padding='same',
                   use_bias=False, name=name_or_none(block_prefix, '_Conv3D'))(x)
        if dropout_rate:
            x = Dropout(dropout_rate)(x)

    return x


def __dense_block(x, nb_layers, nb_filter, growth_rate, bottleneck=False,
                  dropout_rate=None, weight_decay=1e-4, grow_nb_filters=True,
                  return_concat_list=False, block_prefix=None):
    '''
    Build a dense_block where the output of each conv_block is fed
    to subsequent ones
    # Arguments
        x: input keras tensor
        nb_layers: the number of conv_blocks to append to the model
        nb_filter: integer, the dimensionality of the output space
            (i.e. the number output of filters in the convolution)
        growth_rate: growth rate of the dense block
        bottleneck: if True, adds a bottleneck convolution block to
            each conv_block
        dropout_rate: dropout rate
        weight_decay: weight decay factor
        grow_nb_filters: if True, allows number of filters to grow
        return_concat_list: set to True to return the list of
            feature maps along with the actual output
        block_prefix: str, for block unique naming
    # Return
        If return_concat_list is True, returns a list of the output
        keras tensor, the number of filters and a list of all the
        dense blocks added to the keras tensor
        If return_concat_list is False, returns a list of the output
        keras tensor and the number of filters
    '''
    with K.name_scope('DenseBlock'):
        concat_axis = 1 if K.image_data_format() == 'channels_first' else -1

        x_list = [x]

        for i in range(nb_layers):
            cb = __conv_block(x, growth_rate, bottleneck, dropout_rate, weight_decay,
                              block_prefix=name_or_none(block_prefix, '_%i' % i))
            x_list.append(cb)

            x = concatenate([x, cb], axis=concat_axis)

            if grow_nb_filters:
                nb_filter += growth_rate

        if return_concat_list:
            return x, nb_filter, x_list
        else:
            return x, nb_filter


def __transition_block(ip, nb_filter, compression=1.0, weight_decay=1e-4,
                       block_prefix=None, transition_pooling='max'):
    '''
    Adds a pointwise convolution layer (with batch normalization and relu),
    and an average pooling layer. The number of output convolution filters
    can be reduced by appropriately reducing the compression parameter.
    # Arguments
        ip: input keras tensor
        nb_filter: integer, the dimensionality of the output space
            (i.e. the number output of filters in the convolution)
        compression: calculated as 1 - reduction. Reduces the number
            of feature maps in the transition block.
        weight_decay: weight decay factor
        block_prefix: str, for block unique naming
    # Input shape
        4D tensor with shape:
        `(samples, channels, rows, cols)` if data_format='channels_first'
        or 4D tensor with shape:
        `(samples, rows, cols, channels)` if data_format='channels_last'.
    # Output shape
        4D tensor with shape:
        `(samples, nb_filter * compression, rows / 2, cols / 2)`
        if data_format='channels_first'
        or 4D tensor with shape:
        `(samples, rows / 2, cols / 2, nb_filter * compression)`
        if data_format='channels_last'.
    # Returns
        a keras tensor
    '''
    with K.name_scope('Transition'):
        concat_axis = 1 if K.image_data_format() == 'channels_first' else -1

        x = BatchNormalization(axis=concat_axis, epsilon=1.1e-5,
                               name=name_or_none(block_prefix, '_bn'))(ip)
        x = Activation('relu')(x)
        x = Conv3D(int(nb_filter * compression), (1, 1, 1), kernel_initializer='he_normal',
                   padding='same', use_bias=False, kernel_regularizer=l2(weight_decay),
                   name=name_or_none(block_prefix, '_Conv3D'))(x)
        if transition_pooling == 'avg':
            print(x.shape)
            if x.shape[3] >= 2:
              x = AveragePooling3D((2, 2, 2), strides=(2, 2, 2))(x)   ### <---- modifying from 2 to 1 
            else:
              x = AveragePooling3D((2, 2, 1), strides=(2, 2, 1))(x)   ### <---- modifying from 2 to 1 
        elif transition_pooling == 'max':
            if x.shape[3] >= 2:
              x = MaxPooling3D((2, 2, 2), strides=(2, 2, 2))(x)
            else:
              x = MaxPooling3D((2, 2, 1), strides=(2, 2, 1))(x)
        return x


def __transition_up_block(ip, nb_filters, type='deconv', weight_decay=1E-4,
                          block_prefix=None):
    '''Adds an upsampling block. Upsampling operation relies on the the type parameter.
    # Arguments
        ip: input keras tensor
        nb_filters: integer, the dimensionality of the output space
            (i.e. the number output of filters in the convolution)
        type: can be 'upsampling', 'subpixel', 'deconv'. Determines
            type of upsampling performed
        weight_decay: weight decay factor
        block_prefix: str, for block unique naming
    # Input shape
        4D tensor with shape:
        `(samples, channels, rows, cols)` if data_format='channels_first'
        or 4D tensor with shape:
        `(samples, rows, cols, channels)` if data_format='channels_last'.
    # Output shape
        4D tensor with shape:
        `(samples, nb_filter, rows * 2, cols * 2)` if data_format='channels_first'
        or 4D tensor with shape:
        `(samples, rows * 2, cols * 2, nb_filter)` if data_format='channels_last'.
    # Returns
        a keras tensor
    '''
    with K.name_scope('TransitionUp'):

        if type == 'upsampling':
            x = UpSampling3D(name=name_or_none(block_prefix, '_upsampling'))(ip)
        elif type == 'subpixel':
            x = Conv3D(nb_filters, (3, 3, 3), activation='relu', padding='same',
                       kernel_regularizer=l2(weight_decay), use_bias=False,
                       kernel_initializer='he_normal',
                       name=name_or_none(block_prefix, '_Conv3D'))(ip)
            x = SubPixelUpscaling(scale_factor=2,
                                  name=name_or_none(block_prefix, '_subpixel'))(x)
            x = Conv3D(nb_filters, (3, 3, 3), activation='relu', padding='same',
                       kernel_regularizer=l2(weight_decay), use_bias=False,
                       kernel_initializer='he_normal',
                       name=name_or_none(block_prefix, '_Conv3D'))(x)
        else:
            x = Conv3DTranspose(nb_filters, (3, 3, 3), activation='relu', padding='same',
                                strides=(2, 2, 2), kernel_initializer='he_normal',
                                kernel_regularizer=l2(weight_decay),
                                name=name_or_none(block_prefix, '_Conv3DT'))(ip)
        return x


def __create_dense_net(nb_classes, img_input, include_top, depth=40, nb_dense_block=3,
                       growth_rate=12, nb_filter=-1, nb_layers_per_block=-1,
                       bottleneck=False, reduction=0.0, dropout_rate=None,
                       weight_decay=1e-4, subsample_initial_block=False, pooling=None,
                       activation='softmax', transition_pooling='avg'):
    ''' Build the DenseNet model
    # Arguments
        nb_classes: number of classes
        img_input: tuple of shape (channels, rows, columns) or (rows, columns, channels)
        include_top: flag to include the final Dense layer
        depth: number or layers
        nb_dense_block: number of dense blocks to add to end (generally = 3)
        growth_rate: number of filters to add per dense block
        nb_filter: initial number of filters. Default -1 indicates initial number
            of filters is 2 * growth_rate
        nb_layers_per_block: number of layers in each dense block.
                Can be a -1, positive integer or a list.
                If -1, calculates nb_layer_per_block from the depth of the network.
                If positive integer, a set number of layers per dense block.
                If list, nb_layer is used as provided. Note that list size must
                be (nb_dense_block + 1)
        bottleneck: add bottleneck blocks
        reduction: reduction factor of transition blocks. Note : reduction value is
            inverted to compute compression
        dropout_rate: dropout rate
        weight_decay: weight decay rate
        subsample_initial_block: Changes model type to suit different datasets.
            Should be set to True for ImageNet, and False for CIFAR datasets.
            When set to True, the initial convolution will be strided and
            adds a MaxPooling3D before the initial dense block.
        pooling: Optional pooling mode for feature extraction
            when `include_top` is `False`.
            - `None` means that the output of the model
                will be the 4D tensor output of the
                last convolutional layer.
            - `avg` means that global average pooling
                will be applied to the output of the
                last convolutional layer, and thus
                the output of the model will be a
                2D tensor.
            - `max` means that global max pooling will
                be applied.
        activation: Type of activation at the top layer. Can be one of 'softmax' or
            'sigmoid'. Note that if sigmoid is used, classes must be 1.
        transition_pooling: `avg` for avg pooling (default), `max` for max pooling,
            None for no pooling during scale transition blocks. Please note that this
            default differs from the DenseNetFCN paper in accordance with the DenseNet
            paper.
    # Returns
        a keras tensor
    # Raises
        ValueError: in case of invalid argument for `reduction`
            or `nb_dense_block`
    '''
    with K.name_scope('DenseNet'):
        concat_axis = 1 if K.image_data_format() == 'channels_first' else -1

        if reduction != 0.0:
            if not (reduction <= 1.0 and reduction > 0.0):
                raise ValueError('`reduction` value must lie between 0.0 and 1.0')

        # layers in each dense block
        if type(nb_layers_per_block) is list or type(nb_layers_per_block) is tuple:
            nb_layers = list(nb_layers_per_block)  # Convert tuple to list

            if len(nb_layers) != nb_dense_block:
                raise ValueError('If `nb_dense_block` is a list, its length must match '
                                 'the number of layers provided by `nb_layers`.')

            final_nb_layer = nb_layers[-1]
            nb_layers = nb_layers[:-1]
        else:
            if nb_layers_per_block == -1:
                assert (depth - 4) % 3 == 0, ('Depth must be 3 N + 4 '
                                              'if nb_layers_per_block == -1')
                count = int((depth - 4) / 3)

                if bottleneck:
                    count = count // 2

                nb_layers = [count for _ in range(nb_dense_block)]
                final_nb_layer = count
            else:
                final_nb_layer = nb_layers_per_block
                nb_layers = [nb_layers_per_block] * nb_dense_block

        # compute initial nb_filter if -1, else accept users initial nb_filter
        if nb_filter <= 0:
            nb_filter = 2 * growth_rate

        # compute compression factor
        compression = 1.0 - reduction

        # Initial convolution
        if subsample_initial_block:
            initial_kernel = (7, 7, 7)
            initial_strides = (2, 2, 2)
        else:
            initial_kernel = (3, 3, 3)
            initial_strides = (1, 1, 1)

        x = Conv3D(nb_filter, initial_kernel, kernel_initializer='he_normal',
                   padding='same', name='initial_Conv3D', strides=initial_strides,
                   use_bias=False, kernel_regularizer=l2(weight_decay))(img_input)

        if subsample_initial_block:
            x = BatchNormalization(axis=concat_axis, epsilon=1.1e-5,
                                   name='initial_bn')(x)
            x = Activation('relu')(x)
            x = MaxPooling3D((3, 3, 3), strides=(2, 2, 2), padding='same')(x)

        # Add dense blocks
        for block_idx in range(nb_dense_block - 1):
            x, nb_filter = __dense_block(x, nb_layers[block_idx], nb_filter,
                                         growth_rate, bottleneck=bottleneck,
                                         dropout_rate=dropout_rate,
                                         weight_decay=weight_decay,
                                         block_prefix='dense_%i' % block_idx)
            # add transition_block
            x = __transition_block(x, nb_filter, compression=compression,
                                   weight_decay=weight_decay,
                                   block_prefix='tr_%i' % block_idx,
                                   transition_pooling=transition_pooling)
            nb_filter = int(nb_filter * compression)

        # The last dense_block does not have a transition_block
        x, nb_filter = __dense_block(x, final_nb_layer, nb_filter, growth_rate,
                                     bottleneck=bottleneck, dropout_rate=dropout_rate,
                                     weight_decay=weight_decay,
                                     block_prefix='dense_%i' % (nb_dense_block - 1))

        x = BatchNormalization(axis=concat_axis, epsilon=1.1e-5, name='final_bn')(x)
        x = Activation('relu')(x)

        if include_top:
            if pooling == 'avg':
                x = GlobalAveragePooling3D()(x)
            elif pooling == 'max':
                x = GlobalMaxPooling3D()(x)
            x = Dense(nb_classes, activation=activation)(x)
        else:
            if pooling == 'avg':
                x = GlobalAveragePooling3D()(x)
            elif pooling == 'max':
                x = GlobalMaxPooling3D()(x)

        return x



In [0]:
def DenseNet3D(input_shape=None,
               depth=40,
               nb_dense_block=3,
               growth_rate=12,
               nb_filter=-1,
               nb_layers_per_block=-1,
               bottleneck=False,
               reduction=0.0,
               dropout_rate=0.0,
               weight_decay=1e-4,
               subsample_initial_block=False,
               include_top=True,
               input_tensor=None,
               pooling=None,
               classes=10,
               activation='softmax',
               transition_pooling='avg'):
    '''Instantiate the DenseNet architecture.
    The model and the weights are compatible with both
    TensorFlow and Theano. The dimension ordering
    convention used by the model is the one
    specified in your Keras config file.
    # Arguments
        input_shape: optional shape tuple, only to be specified
            if `include_top` is False (otherwise the input shape
            has to be `(224, 224, 224, 3)` (with `channels_last` dim ordering)
            or `(3, 224, 224, 224)` (with `channels_first` dim ordering).
            It should have exactly 4 inputs channels,
            and width and height should be no smaller than 8.
            E.g. `(224, 224, 224, 3)` would be one valid value.
        depth: number or layers in the DenseNet
        nb_dense_block: number of dense blocks to add to end
        growth_rate: number of filters to add per dense block
        nb_filter: initial number of filters. -1 indicates initial
            number of filters will default to 2 * growth_rate
        nb_layers_per_block: number of layers in each dense block.
            Can be a -1, positive integer or a list.
            If -1, calculates nb_layer_per_block from the network depth.
            If positive integer, a set number of layers per dense block.
            If list, nb_layer is used as provided. Note that list size must
            be nb_dense_block
        bottleneck: flag to add bottleneck blocks in between dense blocks
        reduction: reduction factor of transition blocks.
            Note : reduction value is inverted to compute compression.
        dropout_rate: dropout rate
        weight_decay: weight decay rate
        subsample_initial_block: Changes model type to suit different datasets.
            Should be set to True for ImageNet, and False for CIFAR datasets.
            When set to True, the initial convolution will be strided and
            adds a MaxPooling3D before the initial dense block.
        include_top: whether to include the fully-connected
            layer at the top of the network.
        input_tensor: optional Keras tensor (i.e. output of `layers.Input()`)
            to use as image input for the model.
        pooling: Optional pooling mode for feature extraction
            when `include_top` is `False`.
            - `None` means that the output of the model
                will be the 4D tensor output of the
                last convolutional layer.
            - `avg` means that global average pooling
                will be applied to the output of the
                last convolutional layer, and thus
                the output of the model will be a
                2D tensor.
            - `max` means that global max pooling will
                be applied.
        classes: optional number of classes to classify images
            into, only to be specified if `include_top` is True.
        activation: Type of activation at the top layer. Can be one of
            'softmax' or 'sigmoid'. Note that if sigmoid is used,
             classes must be 1.
        transition_pooling: `avg` for avg pooling (default), `max` for max pooling,
            None for no pooling during scale transition blocks. Please note that this
            default differs from the DenseNetFCN paper in accordance with the DenseNet
            paper.
    # Returns
        A Keras model instance.
    # Raises
        ValueError: in case of invalid input shape.
    '''

    if activation not in ['softmax', 'sigmoid']:
        raise ValueError('activation must be one of "softmax" or "sigmoid"')

    if activation == 'sigmoid' and classes != 1:
        raise ValueError('sigmoid activation can only be used when classes = 1')

    if input_tensor is None:
        img_input = Input(shape=input_shape)
    else:
        if not K.is_keras_tensor(input_tensor):
            img_input = Input(tensor=input_tensor, shape=input_shape)
        else:
            img_input = input_tensor

    x = __create_dense_net(classes, img_input, include_top, depth, nb_dense_block,
                           growth_rate, nb_filter, nb_layers_per_block, bottleneck,
                           reduction, dropout_rate, weight_decay,
                           subsample_initial_block, pooling, activation,
                           transition_pooling)

    # Ensure that the model takes into account
    # any potential predecessors of `input_tensor`.
    if input_tensor is not None:
        inputs = get_source_inputs(input_tensor)
    else:
        inputs = img_input
    # Create model.
    model = Model(inputs, x, name='densenet')

    return model

In [0]:
def DenseNet3DImageNet121(input_shape=None,
                          bottleneck=True,
                          reduction=0.5,
                          dropout_rate=0.0,
                          weight_decay=1e-4,
                          include_top=True,
                          input_tensor=None,
                          pooling=None,
                          classes=1000,
                          activation='softmax'):
    return DenseNet3D(input_shape, depth=121, nb_dense_block=4, growth_rate=32,
                      nb_filter=64, nb_layers_per_block=[6, 12, 24, 16],
                      bottleneck=bottleneck, reduction=reduction,
                      dropout_rate=dropout_rate, weight_decay=weight_decay,
                      subsample_initial_block=True, include_top=include_top,
                      input_tensor=input_tensor,
                      pooling=pooling, classes=classes, activation=activation)


def DenseNet3DImageNet169(input_shape=None,
                          bottleneck=True,
                          reduction=0.5,
                          dropout_rate=0.0,
                          weight_decay=1e-4,
                          include_top=True,
                          input_tensor=None,
                          pooling=None,
                          classes=1000,
                          activation='softmax'):
    return DenseNet3D(input_shape, depth=169, nb_dense_block=4, growth_rate=32,
                      nb_filter=64, nb_layers_per_block=[6, 12, 32, 32],
                      bottleneck=bottleneck, reduction=reduction,
                      dropout_rate=dropout_rate, weight_decay=weight_decay,
                      subsample_initial_block=True, include_top=include_top,
                      input_tensor=input_tensor,
                      pooling=pooling, classes=classes, activation=activation)



###  ** All above saved into DenseNet3D.py **


### load Google Drive

In [0]:
from google.colab import drive
drive.mount('/content/gdrive')

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).


In [0]:
import os
os.chdir('gdrive/My Drive/Colab Notebooks/ProstateX_trial/DenseNet')

In [0]:
import argparse
import os
import sys
import keras

Using TensorFlow backend.


### A one-time data statistics is required for standardizing the dataset

In [0]:
### one-time execution
os.chdir('../raw data/img_patches')
import numpy as np

arr_dict = dict()
for rt, directs, files in os.walk(os.getcwd(), topdown = True):
    if rt == os.getcwd():
        patch_list = directs
        for i in patch_list:
            arr_dict[i] = {}
            
    if directs == []:
        temp_list = []
        for file in files:
            temp_list.append(np.load(os.path.join(rt,file)))
        tmp_array = np.array(temp_list)
        print(tmp_array.shape)
        mean, std = np.empty(tmp_array.shape[-1]), np.empty(tmp_array.shape[-1])
        
        for idx in range(tmp_array.shape[-1]):
            mean[idx] = np.mean(tmp_array[:,:,:,:,idx])
            std[idx] = np.std(tmp_array[:,:,:,:,idx])
            
        for key in arr_dict.keys():
            if key in rt:
                arr_dict[key] = [mean, std]
        
        break
                
print(arr_dict)

### losses.py (for focal loss function)

In [0]:
from losses import categorical_focal_loss

def model_with_weights(model, weights, skip_mismatch):
    """ Load weights for model.

    Args
        model         : The model to load weights for.
        weights       : The weights to load.
        skip_mismatch : If True, skips layers whose shape of weights doesn't match with the model.
    """
    if weights is not None:
        # loads the weights of the model from a hdf5 file (created by save_weights)
        model.load_weights(weights, by_name=True, skip_mismatch=skip_mismatch)
    return model



def create_models(args, freeze_backbone=False):  # >>> can use variadic auguments to have more model config options
    """ Creates three models (model, training_model, prediction_model). ??

    *Args
        backbone : A function to call to create a densenet model with a given backbone.
        classes        : The number of classes to train.
        weights            : The weights to load into the model.
        multi_gpu          : The number of GPUs to use for training.
        freeze_backbone    : If True, disables learning for the backbone.

    Returns
        training_model   : The training model. If multi_gpu=0, this is identical to model.
    """
    
    modifier = freeze_model if freeze_backbone else None

    # Keras recommends initialising a multi-gpu model on the CPU to ease weight sharing, and to prevent OOM errors.
    # optionally wrap in a parallel model
    in_shape = tuple([int(x) for x in args.input_shape.lstrip('(').rstrip(')').split(',')])
    if args.generate_model:
        ## generate a densenet model from DenseNet3D
        print('will generate a model from DenseNet3D')
        model = DenseNet3D(in_shape, depth=args.model_depth, nb_dense_block=4, growth_rate=32,
                      nb_filter=-1, nb_layers_per_block=-1,
                      bottleneck=True, reduction=0.5,
                      dropout_rate=0.0, weight_decay=1e-4,
                      subsample_initial_block=True, include_top=True,
                      input_tensor=None,
                      pooling='avg', classes=args.classes, activation='softmax')

        
    else:
        if args.backbone == 'densenet121':
            model = DenseNet3DImageNet121(in_shape, classes = args.classes, pooling = 'avg')
            
        if args.backbone == 'densenet169':
            model = DenseNet3DImageNet169(in_shape, classes = args.classes, pooling = 'avg')
            
    
    if args.multi_gpu_force: 
        from keras.utils import multi_gpu_model
        with tf.device('/cpu:0'): 
            if args.weights is not None:  
                model = model_with_weights(model, weights=args.weights, skip_mismatch=True)
        training_model = multi_gpu_model(model, gpus=args.multi_gpu)
    
    else:
        if args.weights is not None:
            model = model_with_weights(model, weights=args.weights, skip_mismatch=True)
        training_model = model

    optm = OPTS.Adam()  # adam or modified adam??
    # compile model
    training_model.compile(loss=[categorical_focal_loss(alpha=.25, gamma=2)], optimizer = optm, metrics = ['accuracy']) 
    #training_model.compile(loss= 'binary_crossentropy', optimizer=optm, metrics = ['accuracy'])

    return training_model


### Data generators
        train_generator = data_generator(train_dataset, self.config, shuffle=True,
                                         augmentation=augmentation,
                                         batch_size=self.config.BATCH_SIZE,
                                         no_augmentation_sources=no_augmentation_sources)
        val_generator = data_generator(val_dataset, self.config, shuffle=True,
                                       batch_size=self.config.BATCH_SIZE)

        
        
        self.keras_model.fit_generator(
            train_generator,
            initial_epoch=self.epoch,
            epochs=epochs,
            steps_per_epoch=self.config.STEPS_PER_EPOCH,
            callbacks=callbacks,
            validation_data=val_generator,
            validation_steps=self.config.VALIDATION_STEPS,
            max_queue_size=100,
            workers=workers,
            use_multiprocessing=True,

### transforms.py (for image augmentation)

In [0]:
import numpy as np
from keras.preprocessing.image import *


def flip_axis(x, axis):
    x = np.asarray(x).swapaxes(axis, 0)
    x = x[::-1, ...]
    x = x.swapaxes(0, axis)
    
    return x


def random_rotation3D(x, rg, row_axis=1, col_axis=2, slice_axis=3, channel_axis=0,
                    fill_mode='reflect', cval=0., interpolation_order=1):
    """Performs a random rotation of a Numpy image tensor.
    # Arguments
        x: Input tensor. Must be 4D.
        rg: Rotation range, in degrees.
        row_axis: Index of axis for rows in the input tensor.
        col_axis: Index of axis for columns in the input tensor.
        channel_axis: Index of axis for channels in the input tensor.
        fill_mode: Points outside the boundaries of the input
            are filled according to the given mode
            (one of `{'constant', 'nearest', 'reflect', 'wrap'}`).
        cval: Value used for points outside the boundaries
            of the input if `mode='constant'`.
        interpolation_order: int, order of spline interpolation.
            see `ndimage.interpolation.affine_transform`
    # Returns
        Rotated Numpy image tensor.
    """
    X = []
    theta = np.random.uniform(-rg, rg)
    for idx in range(x.shape[slice_axis]):
      X.append(apply_affine_transform(x[:,:,:,idx], theta=theta, channel_axis=channel_axis,
                               fill_mode=fill_mode, cval=cval,
                               order=interpolation_order))
    
    return np.array(X)


def random_shift3D(x, wrg, hrg, row_axis=1, col_axis=2, slice_axis=3, channel_axis=0,
                 fill_mode='reflect', cval=0., interpolation_order=1):
    """Performs a random spatial shift of a Numpy image tensor.
    # Arguments
        x: Input tensor. Must be 4D.
        wrg: Width shift range; should be less than half of delta that is used to investigate overlap in randomly patched images.
        hrg: Height shift range, should be less than half of delta that is used to investigate overlap in randomly patched images.
        row_axis: Index of axis for rows in the input tensor.
        col_axis: Index of axis for columns in the input tensor.
        channel_axis: Index of axis for channels in the input tensor.
        fill_mode: Points outside the boundaries of the input
            are filled according to the given mode
            (one of `{'constant', 'nearest', 'reflect', 'wrap'}`).
        cval: Value used for points outside the boundaries
            of the input if `mode='constant'`.
        interpolation_order: int, order of spline interpolation.
            see `ndimage.interpolation.affine_transform`
    # Returns
        Shifted Numpy image tensor.
    """
    X = []
    #h, w = x.shape[row_axis], x.shape[col_axis]
    tx = np.random.uniform(-hrg, hrg) # * h
    ty = np.random.uniform(-wrg, wrg) # * w
    for idx in range(x.shape[slice_axis]):
      X.append(apply_affine_transform(x[:,:,:,idx], tx=tx, ty=ty, channel_axis=channel_axis,
                               fill_mode=fill_mode, cval=cval,
                               order=interpolation_order))
    
    return np.array(X)


def random_shear3D(x, intensity, row_axis=1, col_axis=2, slice_axis=3, channel_axis=0,
                 fill_mode='reflect', cval=0., interpolation_order=1):
    """Performs a random spatial shear of a Numpy image tensor.
    # Arguments
        x: Input tensor. Must be 4D.
        intensity: Transformation intensity in degrees.
        row_axis: Index of axis for rows in the input tensor.
        col_axis: Index of axis for columns in the input tensor.
        channel_axis: Index of axis for channels in the input tensor.
        fill_mode: Points outside the boundaries of the input
            are filled according to the given mode
            (one of `{'constant', 'nearest', 'reflect', 'wrap'}`).
        cval: Value used for points outside the boundaries
            of the input if `mode='constant'`.
        interpolation_order: int, order of spline interpolation.
            see `ndimage.interpolation.affine_transform`
    # Returns
        Sheared Numpy image tensor.
    """
    X = []
    shear = np.random.uniform(-intensity, intensity)
    for idx in range(x.shape[slice_axis]):
        X.append(apply_affine_transform(x[:,:,:,idx], shear=shear, channel_axis=channel_axis,
                               fill_mode=fill_mode, cval=cval,
                               order=interpolation_order))
    
    return np.array(X)


def random_zoom3D(x, zoom_range, row_axis=1, col_axis=2, slice_axis=3, channel_axis=0,
                fill_mode='reflect', cval=0., interpolation_order=1):
    """Performs a random spatial zoom of a Numpy image tensor.
    # Arguments
        x: Input tensor. Must be 3D.
        zoom_range: Tuple of floats; zoom range for width and height.
        row_axis: Index of axis for rows in the input tensor.
        col_axis: Index of axis for columns in the input tensor.
        channel_axis: Index of axis for channels in the input tensor.
        fill_mode: Points outside the boundaries of the input
            are filled according to the given mode
            (one of `{'constant', 'nearest', 'reflect', 'wrap'}`).
        cval: Value used for points outside the boundaries
            of the input if `mode='constant'`.
        interpolation_order: int, order of spline interpolation.
            see `ndimage.interpolation.affine_transform`
    # Returns
        Zoomed Numpy image tensor.
    # Raises
        ValueError: if `zoom_range` isn't a tuple.
    """
    X = []
    if len(zoom_range) != 2:
        raise ValueError('`zoom_range` should be a tuple or list of two'
                         ' floats. Received: %s' % (zoom_range,))

    if zoom_range[0] == 1 and zoom_range[1] == 1:
        zx, zy = 1, 1
    else:
        zx, zy = np.random.uniform(zoom_range[0], zoom_range[1], 2)
    for idx in range(x.shape[slice_axis]):
        X.append(apply_affine_transform(x[:,:,:,idx], zx=zx, zy=zy, channel_axis=channel_axis,
                               fill_mode=fill_mode, cval=cval,
                               order=interpolation_order))
    
    return np.array(X)


def random_brightness3D(x, brightness_range, slice_axis=3):
    """Performs a random brightness shift.
    # Arguments
        x: Input tensor. Must be 4D.
        brightness_range: Tuple of floats; brightness range.
        channel_axis: Index of axis for channels in the input tensor.
    # Returns
        Numpy image tensor.
    # Raises
        ValueError if `brightness_range` isn't a tuple.
    """
    X = []
    if len(brightness_range) != 2:
        raise ValueError(
            '`brightness_range should be tuple or list of two floats. '
            'Received: %s' % (brightness_range,))

    u = np.random.uniform(brightness_range[0], brightness_range[1])
    for idx in range(x.shape[slice_axis]):
        temp = x[:,:,:,idx].transpose(1,2,0)       
        X.append(apply_brightness_shift(temp, u).transpose(2,0,1))
    
    return np.array(X)


def random_flip3D(x, h_flip_prob =0., v_flip_prob=0., row_axis=1, col_axis=2, slice_axis=3, channel_axis=0):
    """Performs a transformation randomly contain vertical/ horizontal flips (or not) on the X-Y plan
    # Arguments
        x: Input tensor. 
        h_flip_prob: The chance of getting a horizontal flip
        v_flip_prob: The chance of getting a vertical flip
    # Returns
        Numpy image tensor
  
    """
    if np.random.random() < h_flip_prob:
        x = flip_axis(x, col_axis)
    
    if np.random.random() < v_flip_prob:
        x = flip_axis(x, row_axis)
    
    return x


def random_transform_generator(img, r_axis, c_axis, s_axis, ch_axis, trans_pars):
    
    #print(trans_pars)
    if 'rot_rg' in trans_pars.keys():
        img = random_rotation3D(img, rg=trans_pars['rot_rg'], row_axis=r_axis, col_axis=c_axis, slice_axis=s_axis, channel_axis=ch_axis)  
        img = np.moveaxis(img, 0, -1)
                
    if 'shift_rgs' in trans_pars.keys():
        img = random_shift3D(img, trans_pars['shift_rgs'][0], trans_pars['shift_rgs'][1], row_axis=r_axis, col_axis=c_axis, slice_axis=s_axis, channel_axis=ch_axis) 
        img = np.moveaxis(img, 0, -1)
                
    if 'sheer' in trans_pars.keys():
        img = random_shear3D(img, trans_pars['sheer'], row_axis=r_axis, col_axis=c_axis, slice_axis=s_axis, channel_axis=ch_axis)
        img = np.moveaxis(img, 0, -1)
                
    if 'zoom' in trans_pars.keys():
        img = random_zoom3D(img, trans_pars['zoom'], row_axis=r_axis, col_axis=c_axis, slice_axis=s_axis, channel_axis=ch_axis)
        img = np.moveaxis(img, 0, -1)
                
    if 'brightness' in trans_pars.keys():
        img = random_brightness3D(img, trans_pars['brightness'], slice_axis=s_axis)
        img = np.moveaxis(img, 0, -1)
                
    if 'flip' in trans_pars.keys():
        img = random_flip3D(img, h_flip_prob=trans_pars['flip'][0], v_flip_prob=trans_pars['flip'][1], row_axis=r_axis, col_axis=c_axis, slice_axis=s_axis, channel_axis=ch_axis)
                
    return img

## Some preliminary tests

In [0]:
tr_pr ={'rot_rg':15, 'shift_rgs':(4, 4), 'sheer':10, 'zoom': (0.8, 1.1), 'brightness': (0.8, 1.2), 'flip': (0.02, 0.3)}

In [0]:
print(test_img.shape)
from matplotlib import pyplot as plt
for i in range(5):
  ax = plt.subplot(1,5,i+1)
  ax.imshow(test_img[:,:,i,1].T)
plt.show()
test = np.moveaxis(test_img, -1, 0)

NameError: ignored

In [0]:
print(test.shape)
post_img = random_transform_generator(test, 1, 2, 3, 0, tr_pr)

In [0]:
print(post_img.shape)
for i in range(5):
  ax = plt.subplot(1,5,i+1)
  ax.imshow(post_img[1,:,:,i].T)
plt.show()

## loading data generation


In [0]:
os.chdir("../raw data/img_patches")


In [0]:
'''
### check webpage:  https://stanford.edu/~shervine/blog/keras-how-to-generate-data-on-the-fly
#### Create a Generator class ####
class Generator(keras.utils.Sequence):
    def __init__(self, list_IDs, batch_size = 32):
        self.batch_size = batch_size
        self.list_IDs
        self.shuffle = shuffle
        self.on_epoch_end()
        
    def on_epoch_end(self):
        #updates indexes after each epoch
        self.indexes = np.arange(len(self.list_IDs))
        if self.shuffle == True:
            np.random.shuffle(self.indexes)
            
    def __getitem__(self, index):
'''

class Generator(keras.utils.Sequence):
    def __init__(
        self, 
        list_IDs, #entire train/ validation dataset IDs
        df,  # patch info DataFrame
        dim,      #image patch dimensions (H,W,D)
        n_channels,  # number of channels (image modalities)
        com_args,  # other common arguments
        transform_generator = False, #True for image augmentatio
        batch_size = 16,
        shuffle = True,
        transform_parameters = None,  #if transform_generator is True, transform_parameters required
        
    ):
        """ Initialize Generator object.
        Args
            transform_generator    : A boolin factor determining to activate randomly transformation of images.
            batch_size             : The size of the batches to generate.
            group_method           : Determines how images are grouped together (defaults to 'none', one of ('none', 'entire').
            shuffle_groups         : If True, shuffles the groups each epoch.
            transform_parameters   : The transform parameters used for data augmentation.
        """
        
        self.list_IDs = list_IDs
        self.df = df
        self.dim = dim
        self.channels = n_channels
        self.transform_generator = transform_generator
        self.batch_size = int(batch_size)
        self.shuffle = shuffle
        self.transform_parameters = transform_parameters 
        self.data_folder = '_'.join([str(x) for x in self.dim])
        self.on_epoch_end()
        self.com_args = com_args
        super(Generator, self).__init__()
        
    
    def on_epoch_end(self):
        #updates group indexes after each epoch
        self.indexes = np.arange(len(self.list_IDs))
        if self.shuffle:
            np.random.shuffle(self.indexes)
     
    def __len__(self):
        return int(np.ceil(len(self.list_IDs)/float(self.batch_size)))
      
    
    def __getitem__(self, index):
        #Genereate indexes of the batch
        batch_indexes = self.indexes[index*self.batch_size:(index+1)*self.batch_size]
        #Find list of IDs in the batch
        IDs_in_batch = [self.list_IDs[idx] for idx in batch_indexes]
        # Generate data
        return self.__data_generation(IDs_in_batch)
      
      
    def __data_generation(self, ID_list):
        # Generates data containing batch_size samples --> X: (n_samples, *dim, n_channels)
        X = np.empty((len(ID_list), *self.dim, self.channels))
        y = np.empty(len(ID_list), dtype = int)
        #X = np.empty((self.batch_size, *self.dim, self.channels))   # channel last
        #y = np.empty((self.batch_size), dtype = int)

        if self.transform_generator:
            for idx, ID in enumerate(ID_list):
                # random_transform_generator:  channel first    
                load_x_patch = np.moveaxis(np.load(os.path.join(self.data_folder, ID)), -1, 0) # move the ch_axis from the last to the first
                X[idx,] = np.moveaxis(random_transform_generator(load_x_patch,  r_axis=1, c_axis=2, s_axis=3, ch_axis=0, trans_pars = self.transform_parameters,), 0, -1)
                y[idx] = int(self.df[self.df['File name'] == ID]['label']) ## from labels extracted from pd.DataFrame
                
        else:
            for idx, ID in enumerate(ID_list):
                X[idx,] = np.load(os.path.join(self.data_folder,ID))
                y[idx] = int(self.df[self.df['File name'] == ID]['label']) ## from labels extracted from pd.DataFrame
                

        assert 'n_classes' in self.com_args.keys(), 'common_args does not include "n_classes"'
        #categorical = np.zeros((n, self.com_args['n_classes']), dtype=dtype)
        if len(ID_list) < self.batch_size:
            print('batch size = {}'.format(len(ID_list)))
        return X, keras.utils.to_categorical(y, num_classes = self.com_args['n_classes'])
        

In [0]:
from sklearn.model_selection import train_test_split
import json

'''
  data_gen = tf.keras.preprocessing.image.ImageDataGenerator(
    rotation_range=90,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest')
    
    
  model.fit_generator(data_gen.flow(trainX, trainY, batch_size=..))
'''

### create generators
def create_generators(args, preprocess_image_IDs_path):
    """ Create generators for training and validation.
        Args:   parseargs object containing configuration for generators.
             including -- batch_size, n_classess, input_shape, random_transform, config 
    """
    
    common_args = {
        'batch_size'       : args.batch_size,
        'n_classes'   : args.classes,
        'input_shape'   : [int(x) for x in args.input_shape.split(',')],
        'config' : args.config,
    }

    # ensure preprocess_image_IDs_path is a .pkl file
    assert preprocess_image_IDs_path.endswith('.pkl'), 'File type is incorrect.'
    
    # create random transform parameter setting for augmenting training data
    # e.g. trans_pars ={'rot_rg':15, 'shift_rgs':(4, 4), 'sheer':10, 'zoom': (0.8, 1.1), 'brightness': (0.8, 1.2), 'flip': (0.02, 0.3)}
    if args.random_transform:
        trans_gen = True
        with open(common_args['config'], 'r') as f:
            config = json.load(f)
        trans_pars = {
            'rot_rg': config['rot_range'],
            'shift_rgs': config['shift_range'],
            'sheer': config['sheer_degree'],
            'zoom': config['zoom_factor'],
            'brightness': config['bright_range'],
            'flip': config['flip_prob'],
            }
    #data_folder = '_'.join([str(x) for x in common_args['input_shape']])
    ID_file_list = pickle.load(open(preprocess_image_IDs_path, 'rb'))
    train_list_IDs, val_list_IDs = train_test_split(list(ID_file_list['File name']), train_size=0.8, random_state=42)   # <- way to split train and validation
    
    train_generator = Generator(
        train_list_IDs, 
        df = ID_file_list,
        dim = common_args['input_shape'][:-1],
        n_channels = common_args['input_shape'][-1],
        transform_generator = trans_gen, 
        batch_size = common_args['batch_size'],
        shuffle = True, 
        transform_parameters = trans_pars, 
        com_args = common_args)

    validation_generator = Generator(
        val_list_IDs,
        df = ID_file_list,
        dim = common_args['input_shape'][:-1],
        n_channels = common_args['input_shape'][-1],
        transform_generator = False, 
        batch_size = common_args['batch_size'],
        shuffle = True, 
        transform_parameters = None, 
        com_args = common_args)
    '''
    else:
        raise ValueError('Invalid data type received: {}'.format(args.dataset_type))
    '''
    return train_generator, validation_generator


In [0]:
import pickle

### test create_generator and Generator
config_file = {
    'rot_range': 15,
    'shift_range': (4,4),
    'sheer_degree': 10,
    'zoom_factor': (0.8, 1.1),
    'bright_range': (0.8, 1.2),
    'flip_prob': (0.02, 0.3)
    }

with open('config_file.json','w') as f:
  json.dump(config_file, f)
  
'''
args = {
    'batch_size': 16,
    'n_classes': 2,
    'input_shape': (96,96,7,3),
    'random_transform': True,
    'config': config_file
    }
'''    
img_patch_file_path = os.path.join(os.getcwd(), '96_96_7_img_patches.pkl')
#train_generator, validation_generator = create_generators(args, img_patch_file_path)

In [0]:
def parse_args(args):
    # parse the auguments
    parser = argparse.ArgumentParser(description='Training script for training a DenseNet network.')
    parser.add_argument('dataset_path', type=str, help='Path to dataset directory.')
    
    group = parser.add_mutually_exclusive_group()
    group.add_argument('--snapshot',          help='Resume training from a snapshot.')
    group.add_argument('--imagenet-weights',  help='Initialize the model with pretrained imagenet weights. This is the default behaviour.', action='store_const', const=False, default=False)
    group.add_argument('--weights',           help='Initialize the model with weights from a file.')
    group.add_argument('--no-weights',        help='Don\'t initialize the model with any weights.', dest='imagenet_weights', action='store_const', const=True)

    model_sel = parser.add_mutually_exclusive_group() #required =  True)
    model_sel.add_argument('--backbone',      help='Backbone model used by DenseNet.', choices=['densenet121', 'densenet169'])
    model_sel.add_argument('--generate_model',help='Generate a user-defined DenseNet model', action='store_true')
    
    parser.add_argument('-s', '--input_shape',help='Input shape with channels_last dim ordering. It should have exactly 4 inputs dims')
    parser.add_argument('-d', '--model_depth',help='Number or layers in the DenseNet.', type=int, default=40)
    parser.add_argument('--classes',          help='Number of classes for the task of classification', type=int, default=1)
    parser.add_argument('--batch-size',       help='Size of the batches.', default=1, type=int)
    parser.add_argument('--gpu',              help='Id of the GPU to use (as reported by nvidia-smi).')
    parser.add_argument('--multi-gpu',        help='Number of GPUs to use for parallel processing.', type=int, default=0)
    parser.add_argument('--multi-gpu-force',  help='Extra flag needed to enable (experimental) multi-gpu support.', action='store_true')
    parser.add_argument('--epochs',           help='Number of epochs to train.', type=int, default=50)
    parser.add_argument('--steps',            help='Number of steps per epoch.', type=int, default=248)
    parser.add_argument('--lr',               help='Learning rate.', type=float, default=1e-5)
    parser.add_argument('--snapshot-path',    help='Path to store snapshots of models during training (defaults to \'./snapshots\')', default='./snapshots')
    parser.add_argument('--tensorboard-dir',  help='Log directory for Tensorboard output', default='./logs')
    parser.add_argument('--no-snapshots',     help='Disable saving snapshots.', dest='snapshots', action='store_false')
    #parser.add_argument('--no-evaluation',    help='Disable per epoch evaluation.', dest='evaluation', action='store_false')
    parser.add_argument('--freeze-backbone',  help='Freeze training of backbone layers.', action='store_true')
    parser.add_argument('--random-transform', help='Randomly transform image and annotations.', action='store_true')
    #parser.add_argument('--image-min-side',   help='Rescale the image so the smallest side is min_side.', type=int, default=800)
    #parser.add_argument('--image-max-side',   help='Rescale the image if the largest side is larger than max_side.', type=int, default=1333)
    parser.add_argument('--config',           help='Path to a configuration parameters .ini file.')
    #parser.add_argument('--weighted-average', help='Compute the mAP using the weighted average of precisions among classes.', action='store_true')
    parser.add_argument('--compute-val-loss', help='Compute validation loss during training', dest='compute_val_loss', action='store_true')

    # Fit generator arguments
    parser.add_argument('--multiprocessing',  help='Use multiprocessing in fit_generator.', action='store_true')
    parser.add_argument('--workers',          help='Number of generator workers.', type=int, default=-1)
    parser.add_argument('--max-queue-size',   help='Queue length for multiprocessing workers in fit_generator.', type=int, default=10)
    
    return check_args(parser.parse_args(args))
  

def check_args(parsed_args):
    """ Function to check for inherent contradictions within parsed arguments.
    For example, batch_size < num_gpus
    Intended to raise errors prior to backend initialisation.
    Args
        parsed_args: parser.parse_args()
    Returns
        parsed_args
    """
    if parsed_args.generate_model:
        if parsed_args.input_shape is None:
            raise ValueError(
            "input shape must be specified as 4-element tuple of channel_last dim ordering")
    
    if parsed_args.classes < 1:
        raise ValueError(
                "input number of classes has to be greater than or equal to one for the task of classification")
        
    if parsed_args.multi_gpu > 1 and parsed_args.batch_size < parsed_args.multi_gpu:
        raise ValueError(
            "Batch size ({}) must be equal to or higher than the number of GPUs ({})".format(parsed_args.batch_size,
                                                                                             parsed_args.multi_gpu))

    if parsed_args.multi_gpu > 1 and parsed_args.snapshot:
        raise ValueError(
            "Multi GPU training ({}) and resuming from snapshots ({}) is not supported.".format(parsed_args.multi_gpu,
                                                                                                parsed_args.snapshot))

    if parsed_args.multi_gpu > 1 and not parsed_args.multi_gpu_force:
        raise ValueError("Multi-GPU support is experimental, use at own risk! Run with --multi-gpu-force if you wish to continue.")

    return parsed_args


In [0]:
args = ['.', '--generate_model', '--classes=2', '--model_depth=88', '--input_shape=96, 96, 7, 3', '--gpu=gpu0', '--multi-gpu=1', '--batch-size=32', '--compute-val-loss', '--multiprocessing', '--config=config_file.json', '--random-transform']

In [0]:
class RedirectModel(keras.callbacks.Callback):
    """Callback which wraps another callback, but executed on a different model.
    ```python
    model = keras.models.load_model('model.h5')
    model_checkpoint = ModelCheckpoint(filepath='snapshot.h5')
    parallel_model = multi_gpu_model(model, gpus=2)
    parallel_model.fit(X_train, Y_train, callbacks=[RedirectModel(model_checkpoint, model)])
    ```
    Args
        callback : callback to wrap.
        model    : model to use when executing callbacks.
    """

    def __init__(self,
                 callback,
                 model):
        super(RedirectModel, self).__init__()

        self.callback = callback
        self.redirect_model = model

    def on_epoch_begin(self, epoch, logs=None):
        self.callback.on_epoch_begin(epoch, logs=logs)

    def on_epoch_end(self, epoch, logs=None):
        self.callback.on_epoch_end(epoch, logs=logs)

    def on_batch_begin(self, batch, logs=None):
        self.callback.on_batch_begin(batch, logs=logs)

    def on_batch_end(self, batch, logs=None):
        self.callback.on_batch_end(batch, logs=logs)

    def on_train_begin(self, logs=None):
        # overwrite the model with our custom model
        self.callback.set_model(self.redirect_model)

        self.callback.on_train_begin(logs=logs)

    def on_train_end(self, logs=None):
        self.callback.on_train_end(logs=logs)

In [0]:
def makedirs(path):
    # Intended behavior: try to create the directory
    # passes if the directory existing; fails otherwise
    try:
        os.makedirs(path)
    except OSError:
        if not os.path.isdir(path):
          raise

In [0]:
def create_callbacks(model, args): # prediction_model, validation_generator, args):
    """ Creates the callbacks to use during training.

    Args
        model: The model used for training.
        prediction_model: The model that should be used for validation.
        validation_generator: The generator for creating validation data.
        args: parseargs args object.

    Returns:
        A list of callbacks used for training.
    """
    callbacks = []
    # tensorboard call back
    tensorboard_callback = None

    if args.tensorboard_dir:
        tensorboard_callback = keras.callbacks.TensorBoard(
            log_dir                = args.tensorboard_dir,
            histogram_freq         = 0,
            batch_size             = args.batch_size,
            write_graph            = True,
            write_grads            = False,
            write_images           = False,
            embeddings_freq        = 0,
            embeddings_layer_names = None,
            embeddings_metadata    = None
        )
        callbacks.append(tensorboard_callback)
        
    '''    
    ### it seems unncessary to keep the evalutaion callback 
    ### evaluation should be done in .fit_generator and/ or .evaluate_generator
    # evaluation callback
    if args.evaluation and validation_generator:
        if args.dataset_type == 'coco':
            from ..callbacks.coco import CocoEval

            # use prediction model for evaluation
            evaluation = CocoEval(validation_generator, tensorboard=tensorboard_callback)
        else:
            evaluation = Evaluate(validation_generator, tensorboard=tensorboard_callback, weighted_average=args.weighted_average)
        evaluation = RedirectModel(evaluation, prediction_model)
        callbacks.append(evaluation)
    '''
    
    # save the model (checkpoint callback)
    if args.snapshots:
        # ensure directory created first; otherwise h5py will error after epoch.
        makedirs(args.snapshot_path)
        checkpoint = keras.callbacks.ModelCheckpoint(
            os.path.join(
                args.snapshot_path,
                '{backbone}_{{epoch:02d}}_{{val_loss:.2f}}.h5'.format(backbone=args.backbone)
            ),
            monitor='val_loss',
            verbose=1,
            save_best_only=False,

            mode='min'
        )
        checkpoint = RedirectModel(checkpoint, model)
        callbacks.append(checkpoint)

    callbacks.append(keras.callbacks.ReduceLROnPlateau(
        monitor  = 'val_loss',
        factor   = 0.5,
        patience = 4,
        verbose  = 1,
        mode     = 'auto',
        min_delta  = 0.0001,
        cooldown = 0,
        min_lr   = 0.0001
    ))
    
    callbacks.append(keras.callbacks.EarlyStopping(
        monitor = 'val_loss', 
        min_delta = 1e-6, 
        patience = 6, 
        verbose = 1, 
        mode = 'auto',
        restore_best_weights = True
    ))

    return callbacks

In [0]:
def check_keras_version(min_ver):
  #minimum_keras_version = 2, 2, 5

  detected = keras.__version__
  required = '.'.join(map(str, min_ver))
  try:
    assert(tuple(map(int, detected.split('.'))) >= min_ver), 'You are using keras version {}. The minimum required version is {}.'.format(detected, required)
  except AssertionError as e:
    print(e, file=sys.stderr)
    #sys.exit(1)

In [0]:
import tensorflow as tf

def get_session():
    """ Construct a modified tf session.
    """
    config = tf.ConfigProto()
    config.gpu_options.allow_growth = True
    return tf.Session(config=config)

In [0]:
## for test purpose
def main(args = None):

    # parse arguments
    if args is None:
        args = sys.argv[1:]
    args = parse_args(args)
    print(args)
    
    # create object that stores backbone information
    #backbone = models.backbone(args.backbone)

    # make sure keras is the minimum required version
    check_keras_version((2,2,4))

    # optionally choose specific GPU
    if args.gpu:
        os.environ['CUDA_VISIBLE_DEVICES'] = args.gpu
    keras.backend.tensorflow_backend.set_session(get_session())

    # optionally load config parameters
    if args.config:
        args.config = args.config #read_config_file(args.config) # <<<  nothing in config;  need to do later
    
    # create the generators  << final step before doing fitting
    train_generator, validation_generator = create_generators(args, img_patch_file_path)
    
    # create the model
    if args.snapshot is not None:
        print('Loading the model, which will take a while ...')
        model = keras.models.load_model(args.snapshot) # need to specify the .hdf5 file path, backbone_name = args.backbone)
    else:
        weights = args.weights
        if weights is None:
            print('Creating model; this may take a while...')
            
            model = create_models(args = args)  ## <-  need to work on "create_models"
            # create_models(backbone, num_classes, weights = None, multi_gpu=0, freeze_backbone=False)
            model.summary()
    
    # create the callbacks
    callbacks = create_callbacks(model, args)
    
    if not args.compute_val_loss:
        valdation_generator = None   # <-- not sure if it is necessary
        
    #start training
    return model.fit_generator(
        generator = train_generator,
        steps_per_epoch = args.steps,
        epochs = args.epochs,
        verbose = 1,
        callbacks = callbacks,
        validation_data = validation_generator,
        validation_steps = 62, 
        max_queue_size=args.max_queue_size,
    )
    
    
if __name__ == '__main__':
  args = args
  main(args)

Namespace(backbone=None, batch_size=32, classes=2, compute_val_loss=True, config='config_file.json', dataset_path='.', epochs=50, freeze_backbone=False, generate_model=True, gpu='gpu0', imagenet_weights=False, input_shape='96, 96, 7, 3', lr=1e-05, max_queue_size=10, model_depth=88, multi_gpu=1, multi_gpu_force=False, multiprocessing=True, random_transform=True, snapshot=None, snapshot_path='./snapshots', snapshots=True, steps=248, tensorboard_dir='./logs', weights=None, workers=-1)
Creating model; this may take a while...
will generate a model from DenseNet3D




(?, 24, 24, 2, 256)
(?, 12, 12, 1, 352)
(?, 6, 6, 1, 400)


Model: "densenet"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
input_1 (InputLayer)            (None, 96, 96, 7, 3) 0                                            
_______________________________________________________

### simple model configuration test

In [0]:
  def main(args = None):
 

    # parse arguments
    if args is None:
        args = sys.argv[1:]
    args = parse_args(args)
 
    # create the model
    model = create_models(backbone = None, num_classes = 2)
    model.summary()
    
    # create the callbacks
    callbacks = create_callbacks(model, args)
    
    #start training
    ### generator: A generator or an instance of 'keras.utils.Sequene' object in order to avoid duplcate data when using multiprocessing
    ### fit_generator(generator, 
    ###               steps_per_epoch=None, 
    ###               epochs=1, 
    ###               verbose=1, 
    ###               callbacks=None, 
    ###               validation_data=None, 
    ###               validation_steps=None, 
    ###               validation_freq=1, 
    ###               class_weight=None, 
    ###               max_queue_size=10, 
    ###               workers=1, 
    ###               use_multiprocessing=False, 
    ###               shuffle=True, 
    ###               initial_epoch=0)
    ###
    history=training_model.fit_generator(
        generator=train_generator,
        steps_per_epoch=args.steps,
        epochs=args.epochs,
        verbose=1,
        callbacks=callbacks,
    )
  

if __name__ == '__main__':
    main()

In [0]:
if __name__ == '__main__':
    model = DenseNet3DImageNet121(input_shape=(96, 96, 7, 4),
                          bottleneck=True,
                          reduction=0.5,
                          dropout_rate=0.0,
                          weight_decay=1e-4,
                          include_top=True,
                          input_tensor=None,
                          pooling= 'avg',
                          classes=2,
                          activation='softmax')
    model.summary()

    

### *** below are some tests

In [0]:
import PIL
import os

print('Pillow Version:', PIL.__version__)
print('PIL Version:', PIL.VERSION)

In [0]:
os.getcwd()

In [0]:
# load and show an image with Pillow
from PIL import Image
# load the image

image = Image.open('gdrive/My Drive/Colab Notebooks/ProstateX_trial/DenseNet/opera_house.jpg')
# summarize some details about the image
print(image.format)
print(image.mode)
print(image.size)
# show the image
image.show()




In [0]:
#%%
# load and display an image with Matplotlib
from matplotlib import image
from matplotlib import pyplot
# load image as pixel array
data = image.imread('opera_house.jpg')
# summarize shape of the pixel array
print(data.dtype)
print(data.shape)
# display the array of pixels as an image
pyplot.imshow(data)
pyplot.show()

In [0]:
from keras.preprocessing.image import apply_affine_transform
post_data = apply_affine_transform(data, 180, channel_axis=2)


In [0]:
print(post_data.shape)
pyplot.imshow(post_data)
pyplot.show()

In [0]:
import keras.preprocessing.image as pre_img
pre_img

In [0]:
import numpy as np
test_img = np.load('ProstateX-0000_fid1.npy')
test_img.shape

In [0]:
from keras.preprocessing.image import *

def random_rotation3D(x, rg, row_axis=1, col_axis=2, slice_axis=3, channel_axis=0,
                    fill_mode='nearest', cval=0., interpolation_order=1):
    """Performs a random rotation of a Numpy image tensor.
    # Arguments
        x: Input tensor. Must be 4D.
        rg: Rotation range, in degrees.
        row_axis: Index of axis for rows in the input tensor.
        col_axis: Index of axis for columns in the input tensor.
        channel_axis: Index of axis for channels in the input tensor.
        fill_mode: Points outside the boundaries of the input
            are filled according to the given mode
            (one of `{'constant', 'nearest', 'reflect', 'wrap'}`).
        cval: Value used for points outside the boundaries
            of the input if `mode='constant'`.
        interpolation_order: int, order of spline interpolation.
            see `ndimage.interpolation.affine_transform`
    # Returns
        Rotated Numpy image tensor.
    """
    X = []
    theta = np.random.uniform(-rg, rg)
    for idx in range(x.shape[slice_axis]):
        X.append(apply_affine_transform(x[:,:,:,idx], theta=theta, channel_axis=channel_axis,
                               fill_mode=fill_mode, cval=cval,
                               order=interpolation_order))
    
    return np.array(X)

  

In [0]:
def random_shift3D(x, wrg, hrg, row_axis=1, col_axis=2, slice_axis=3, channel_axis=0,
                 fill_mode='nearest', cval=0., interpolation_order=1):
    """Performs a random spatial shift of a Numpy image tensor.
    # Arguments
        x: Input tensor. Must be 4D.
        wrg: Width shift range, as a float fraction of the width.
        hrg: Height shift range, as a float fraction of the height.
        row_axis: Index of axis for rows in the input tensor.
        col_axis: Index of axis for columns in the input tensor.
        channel_axis: Index of axis for channels in the input tensor.
        fill_mode: Points outside the boundaries of the input
            are filled according to the given mode
            (one of `{'constant', 'nearest', 'reflect', 'wrap'}`).
        cval: Value used for points outside the boundaries
            of the input if `mode='constant'`.
        interpolation_order: int, order of spline interpolation.
            see `ndimage.interpolation.affine_transform`
    # Returns
        Shifted Numpy image tensor.
    """
    X = []
    h, w = x.shape[row_axis], x.shape[col_axis]
    tx = np.random.uniform(-hrg, hrg) * h
    ty = np.random.uniform(-wrg, wrg) * w
    for idx in range(x.shape[slice_axis]):
        X.append(apply_affine_transform(x[:,:,:,idx], tx=tx, ty=ty, channel_axis=channel_axis,
                               fill_mode=fill_mode, cval=cval,
                               order=interpolation_order))
    
    return np.array(X)

In [0]:
def random_shear3D(x, intensity, row_axis=1, col_axis=2, slice_axis=3, channel_axis=0,
                 fill_mode='nearest', cval=0., interpolation_order=1):
    """Performs a random spatial shear of a Numpy image tensor.
    # Arguments
        x: Input tensor. Must be 4D.
        intensity: Transformation intensity in degrees.
        row_axis: Index of axis for rows in the input tensor.
        col_axis: Index of axis for columns in the input tensor.
        channel_axis: Index of axis for channels in the input tensor.
        fill_mode: Points outside the boundaries of the input
            are filled according to the given mode
            (one of `{'constant', 'nearest', 'reflect', 'wrap'}`).
        cval: Value used for points outside the boundaries
            of the input if `mode='constant'`.
        interpolation_order: int, order of spline interpolation.
            see `ndimage.interpolation.affine_transform`
    # Returns
        Sheared Numpy image tensor.
    """
    X = []
    shear = np.random.uniform(-intensity, intensity)
    for idx in range(x.shape[slice_axis]):
        X.append(apply_affine_transform(x[:,:,:,idx], shear=shear, channel_axis=channel_axis,
                               fill_mode=fill_mode, cval=cval,
                               order=interpolation_order))
    
    return np.array(X)

In [0]:
def random_zoom3D(x, zoom_range, row_axis=1, col_axis=2, slice_axis=3, channel_axis=0,
                fill_mode='nearest', cval=0., interpolation_order=1):
    """Performs a random spatial zoom of a Numpy image tensor.
    # Arguments
        x: Input tensor. Must be 3D.
        zoom_range: Tuple of floats; zoom range for width and height.
        row_axis: Index of axis for rows in the input tensor.
        col_axis: Index of axis for columns in the input tensor.
        channel_axis: Index of axis for channels in the input tensor.
        fill_mode: Points outside the boundaries of the input
            are filled according to the given mode
            (one of `{'constant', 'nearest', 'reflect', 'wrap'}`).
        cval: Value used for points outside the boundaries
            of the input if `mode='constant'`.
        interpolation_order: int, order of spline interpolation.
            see `ndimage.interpolation.affine_transform`
    # Returns
        Zoomed Numpy image tensor.
    # Raises
        ValueError: if `zoom_range` isn't a tuple.
    """
    X = []
    if len(zoom_range) != 2:
        raise ValueError('`zoom_range` should be a tuple or list of two'
                         ' floats. Received: %s' % (zoom_range,))

    if zoom_range[0] == 1 and zoom_range[1] == 1:
        zx, zy = 1, 1
    else:
        zx, zy = np.random.uniform(zoom_range[0], zoom_range[1], 2)
    for idx in range(x.shape[slice_axis]):
        X.append(apply_affine_transform(x[:,:,:,idx], zx=zx, zy=zy, channel_axis=channel_axis,
                               fill_mode=fill_mode, cval=cval,
                               order=interpolation_order))
    
    return np.array(X)


In [0]:
def flip_axis(x, axis):
    x = np.asarray(x).swapaxes(axis, 0)
    x = x[::-1, ...]
    x = x.swapaxes(0, axis)
    
    return x

In [0]:
def random_brightness3D(x, brightness_range, slice_axis=3):
    """Performs a random brightness shift.
    # Arguments
        x: Input tensor. Must be 4D.
        brightness_range: Tuple of floats; brightness range.
        channel_axis: Index of axis for channels in the input tensor.
    # Returns
        Numpy image tensor.
    # Raises
        ValueError if `brightness_range` isn't a tuple.
    """
    X = []
    if len(brightness_range) != 2:
        raise ValueError(
            '`brightness_range should be tuple or list of two floats. '
            'Received: %s' % (brightness_range,))

    u = np.random.uniform(brightness_range[0], brightness_range[1])
    for idx in range(x.shape[slice_axis]):
        temp = x[:,:,:,idx].transpose(1,2,0)       
        X.append(apply_brightness_shift(temp, u).transpose(2,0,1))
    
    return np.array(X)

In [0]:
def random_flip3D(x, h_flip_prob =0., v_flip_prob=0., row_axis=1, col_axis=2, slice_axis=3, channel_axis=0):
    """Performs a transformation randomly contain vertical/ horizontal flips (or not) on the X-Y plan
    # Arguments
        x: Input tensor. 
        h_flip_prob: The chance of getting a horizontal flip
        v_flip_prob: The chance of getting a vertical flip
    # Returns
        Numpy image tensor
  
    """
    if np.random.random() < 1. - h_flip_prob:
        x = flip_axis(x, col_axis)
    
    if np.random.random() < 1. - v_flip_prob:
        x = flip_axis(x, row_axis)
    
    return x

In [0]:
test = np.moveaxis(test_img, 1, -1)
print(test.shape)
from matplotlib import pyplot as plt
for i in range(5):
  ax = plt.subplot(1,5,i+1)
  ax.imshow(test[1,:,:,i].T)
plt.show()

In [0]:
from keras.preprocessing.image import *
#te = random_rotation3D(test, rg = 15)
#te = random_shift3D(test, 0.1, 0.1)
#te = random_shear3D(test,0.02)
#te = random_zoom3D(test,(0.8, 1.25))
te = random_brightness3D(test, (0.5, 1.5))
#te = random_flip3D(test, h_flip_prob = 1.)

In [0]:
print(te.shape)
te_img = np.moveaxis(te, 0, -1)
print(te_img.shape)

In [0]:
for i in range(5):
  ax = plt.subplot(1,5,i+1)
  ax.imshow(te_img[1,:,:,i].T)
plt.show()